In [ ]:
from libraries import *
from sklearn.cross_decomposition import PLSCanonical, PLSRegression, CCA
from sklearn.model_selection import cross_val_score
from numpy import asarray
from numpy import savetxt
from adjustText import adjust_text
import seaborn as sns
from sklearn.cluster import SpectralBiclustering
%matplotlib inline

In [ ]:
deseq2Estimates = pd.read_csv("./TmpOLSOuts/DESeq2DEGenes.csv")
# deseq2Estimates["affectedGene"] = deseq2Estimates.index
# deseq2Estimates = deseq2Estimates[["log2FoldChange", "pvalue", "perturbation", "affectedGene"]]
# deseq2Estimates.columns=["Deseq2_log2FC", "Deseq2_Pval", "perturbation", "affectedGene"]

In [ ]:
deseq2Estimates

In [ ]:
deseq2Perts =deseq2Estimates.affectedGene.value_counts()
deseq2Perts = list(deseq2Perts.index)

In [ ]:
myEstimates = pd.read_csv("./TmpOLSOuts/EffectSizeEstimates.csv")
myEstimates.columns = ["affectedGene", "perturbation", "LogFC", "Pval"]

In [ ]:
myPerts = myEstimates.affectedGene.value_counts()
myPerts = list(myPerts.index)

In [ ]:
deseq2Perts

In [ ]:
len([x for x in deseq2Perts if x not in myPerts])


In [ ]:
len(deseq2Perts)
len(myPerts)

In [ ]:
[x for x in myPerts if x not in deseq2Perts]

In [ ]:
merged = pd.merge(deseq2Estimates, myEstimates, 
                  on=["perturbation", "affectedGene"], how="inner")

# # Group by perturbation and compute correlation
# correlations = (
#     merged.groupby("perturbation")
#     .apply(lambda x: x["Deseq2_log2FC"].corr(x["LogFC"]))
#     .dropna()
# )



In [ ]:
merged.perturbation.value_counts()

In [ ]:
# Plot histogram
plt.figure(figsize=(8, 6))
sns.histplot(correlations, bins=100, stat="count", color="steelblue")
plt.xlabel("Correlation of LogFCs per Perturbation")
plt.ylabel("Number of Perturbations")
plt.title("Distribution of LogFC Correlations")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Read the CSV files with row names as index
coefsAll = pd.read_csv("./TmpOLSOuts/LogFCs.csv", index_col=0)
FDRs = pd.read_csv("./TmpOLSOuts/FDRs.csv", index_col=0)

# Remove specified columns
cols_to_remove = ["const", "log10_n_umis", "mt_frac", "n_genes"]
FDRs.drop(columns=cols_to_remove, inplace=True, errors='ignore')
coefsAll.drop(columns=cols_to_remove, inplace=True, errors='ignore')
coefsAll.shape

In [ ]:
# Select genes with >19 features having FDR < 0.2
s1 = (FDRs < 0.2).sum(axis=0)

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(s1, bins=10000, color='skyblue', edgecolor='black')
plt.xlabel('Number of DE genes with FDR<0.2')
plt.ylabel('Number of perturbations')
plt.title('')
plt.xlim(0,200)
#plt.axvline(x=20, color='red', linestyle='--', linewidth=2)
plt.tight_layout()
plt.show()


In [ ]:
s2 = (FDRs < 0.2).sum(axis=1)

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(s2, bins=100, color='skyblue', edgecolor='black')
plt.xlabel('Number of perturbations a gene been has affected from with FDR<0.2')
plt.ylabel('Number of genes')
plt.title('')
plt.xlim(0,200)
plt.axvline(x=5, color='red', linestyle='--', linewidth=2)
plt.tight_layout()
plt.show()


In [ ]:
selectedPerturbations = s1[s1 > 80].index

# # Subset to selected genes
FDRs = FDRs[selectedPerturbations]
coefsAll = coefsAll[selectedPerturbations]

coefsAll.shape

In [ ]:
s2 = (FDRs < 0.2).sum(axis=1)

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(s2, bins=100, color='skyblue', edgecolor='black')
plt.xlabel('Number of perturbations a gene been has affected from with FDR<0.2')
plt.ylabel('Number of genes')
plt.title('')
plt.xlim(0,200)
plt.axvline(x=5, color='red', linestyle='--', linewidth=2)
plt.tight_layout()
plt.show()


In [ ]:
selectedGenes = s2[s2 > 10].index
len(selectedGenes)
FDRs = FDRs.T
coefsAll = coefsAll.T
# # Subset to selected genes
FDRs = FDRs[selectedGenes]
coefsAll = coefsAll[selectedGenes]

coefsAll.shape

Generate guide modules with leiden

In [ ]:
pertAnndat = sc.AnnData(X=np.transpose(coefsAll))
sc.pp.scale(pertAnndat)
sc.pp.pca(pertAnndat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(pertAnndat, n_neighbors=5)
sc.tl.leiden(pertAnndat, resolution=1)
sc.tl.umap(pertAnndat)
pertAnndat.obs["GeneName"] = coefsAll.columns

f, ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(pertAnndat, color='leiden',ax=ax, size=100, 
           legend_fontoutline=3, 
           legend_loc = 'on data',
           legend_fontsize=14, legend_fontweight='normal')

In [ ]:
adata = sc.read("/home/eraslab1/Projects/AbbasScreen/Data/adataSingles.h5ad")

In [ ]:
pertAnndat.obs["leiden"].unique()

In [ ]:
for elem in [,"16","14"]:
    # print(elem)
    geneModuleList=list(pertAnndat.obs.loc[pertAnndat.obs["leiden"] == elem,"GeneName"])
    print(geneModuleList)
    sc.tl.score_genes(adata=adata, gene_list=geneModuleList, score_name=elem)
    sc.pl.umap(adata, color=elem, size=1, color_map="coolwarm", vmax=2)


In [ ]:
nClust=len(pertAnndat.obs["leiden"].unique())
geneModules = pertAnndat.obs[["GeneName", "leiden"]]
geneModules.columns = ["GeneName", "GeneGroup"]
# Ensure 'GeneGroup' is not a MultiIndex or unhashable
geneModules['GeneGroup'] = geneModules['GeneGroup'].astype(str)

# Sort values
geneModules = geneModules.sort_values(["GeneGroup", "GeneName"],
                                      ascending=[True, True])

# Get unique groups safely
unique_groups = sorted(geneModules['GeneGroup'].unique())

# Generate colors from colormap
cmap = plt.get_cmap('tab20')
colors = {group: cmap(i / len(unique_groups)) for i, group in enumerate(unique_groups)}

# Assign group color
geneModules['GroupColor'] = geneModules['GeneGroup'].map(colors)




In [ ]:
tmp = pd.DataFrame(np.corrcoef(coefsAll.transpose()))

tmp.columns = coefsAll.columns
tmp.index = coefsAll.columns
tmp = tmp.loc[geneModules.GeneName, geneModules.GeneName]
sns.clustermap(tmp, row_cluster=False, col_cluster= False, 
               cmap=plt.cm.RdBu, vmin=-0.1, vmax=0.1, 
               row_colors=geneModules.GroupColor, col_colors=geneModules.GroupColor)


In [ ]:
selPert = FDRs.loc[:,['CCN1', 'CYP1A1', 'EDN1', 'FAM230C', 'KRT15', 'KRT17', 'LAMA3', 'MET',
          'PLAU', 'RFX8', 'SLITRK6', 'STEAP4', 'TACSTD2', 'THBS1', 'TM4SF1']]
selPert.shape
s3 = (selPert < 0.2).sum(axis=1)
selKOts = s3[s3>1]

In [ ]:
selFDR = FDRs.loc[selKOts.index, ['CCN1', 'CYP1A1', 'EDN1', 'FAM230C', 'KRT15', 'KRT17', 'LAMA3', 'MET',
          'PLAU', 'RFX8', 'SLITRK6', 'STEAP4', 'TACSTD2', 'THBS1', 'TM4SF1']]
selcoefsAll = coefsAll.loc[selKOts.index, ['CCN1', 'CYP1A1', 'EDN1', 'FAM230C', 'KRT15', 'KRT17', 'LAMA3', 'MET',
          'PLAU', 'RFX8', 'SLITRK6', 'STEAP4', 'TACSTD2', 'THBS1', 'TM4SF1']]

In [ ]:
selcoefsAll[selFDR>0.2] = 0

In [ ]:
selcoefsAll.shape

In [ ]:
sns.set(font_scale=0.6)

sns.clustermap(selcoefsAll, cmap=plt.cm.coolwarm, vmin=-0.5, vmax=0.5)


In [ ]:
geneModulesNew = pd.concat([geneModules.loc[geneModules.GuideGroup=="4",], geneModules.loc[geneModules.GuideGroup=="0",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GuideGroup=="1",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GuideGroup=="5",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GuideGroup=="3",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GuideGroup=="2",]])


geneModules = geneModulesNew

In [ ]:
geneCov

In [ ]:
guideModules["GuideColor"] = ""
guideModules.loc[guideModules["GuideGroup"] == "0", "GuideColor"] = pertAnndat.uns["leiden_colors"][0]
guideModules.loc[guideModules["GuideGroup"] == "1", "GuideColor"] = pertAnndat.uns["leiden_colors"][1]
guideModules.loc[guideModules["GuideGroup"] == "2", "GuideColor"] = pertAnndat.uns["leiden_colors"][2]
guideModules.loc[guideModules["GuideGroup"] == "3", "GuideColor"] = pertAnndat.uns["leiden_colors"][3]
guideModules.loc[guideModules["GuideGroup"] == "4", "GuideColor"] = pertAnndat.uns["leiden_colors"][4]
guideModules.loc[guideModules["GuideGroup"] == "5", "GuideColor"] = pertAnndat.uns["leiden_colors"][5]

In [ ]:
guideModules

In [ ]:
guideModules.GuideGroup.value_counts()

In [ ]:
guideModules.GuideGroup.unique()

In [ ]:
# gGroups = guideModules.GuideGroup.cat.codes
# lut = dict(zip(set(gGroups), sns.hls_palette(len(set(gGroups)), l=0.5, s=0.8)))
# row_colors = pd.DataFrame(gGroups)[0].map(lut)

In [ ]:

# fig = plt.figure(figsize=(8,8))
# ax = fig.add_subplot(111)
# cax = ax.matshow(guideCov, cmap=plt.cm.PRGn, vmin=-0.5, vmax=0.5, )
#fig.colorbar(cax, orientation="horizontal")

In [ ]:
guideModules.to_csv("/home/beraslan/jovian-work/analysisSingle/ME_GuideModules_leiden_"+str(nClust)+"_Modules.csv")

Generate gene modules with leiden

In [ ]:
koGenesAnnDat = sc.AnnData(X=coefsAll.transpose())
sc.pp.pca(koGenesAnnDat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(koGenesAnnDat)
sc.tl.leiden(koGenesAnnDat, resolution=0.8)
sc.tl.umap(koGenesAnnDat)
koGenesAnnDat.obs["EffectedGenes"] = coefsAll.columns

f, ax = plt.subplots(figsize=(8, 8))
sc.pl.umap(koGenesAnnDat, color='leiden',ax=ax, size=100, palette = 'Dark2', legend_fontoutline=3, legend_fontsize=14,legend_loc='right margin', legend_fontweight='normal')

In [ ]:
nClust=len(koGenesAnnDat.obs["leiden"].unique())
geneModules = koGenesAnnDat.obs[["EffectedGenes", "leiden"]]
geneModules.columns = ["GeneName", "GeneGroup"]
geneModules

In [ ]:
geneModules = geneModules.sort_values(["GeneGroup", "GeneName"], ascending = (True, True))

In [ ]:
geneModules

In [ ]:
dark2Palette = ["#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E", "#E6AB02", "#A6761D", "#666666"]
geneModules["GeneColor"] = ""
geneModules.loc[geneModules["GeneGroup"] == "0", "GeneColor"] = dark2Palette[0]
geneModules.loc[geneModules["GeneGroup"] == "1", "GeneColor"] = dark2Palette[1]
geneModules.loc[geneModules["GeneGroup"] == "2", "GeneColor"] = dark2Palette[2]
geneModules.loc[geneModules["GeneGroup"] == "3", "GeneColor"] = dark2Palette[3]
geneModules.loc[geneModules["GeneGroup"] == "4", "GeneColor"] = dark2Palette[4]
geneModules.loc[geneModules["GeneGroup"] == "5", "GeneColor"] = dark2Palette[5]
geneModules.loc[geneModules["GeneGroup"] == "6", "GeneColor"] = dark2Palette[6]
geneModules.loc[geneModules["GeneGroup"] == "7", "GeneColor"] = dark2Palette[7]

In [ ]:
#geneModulesNew = pd.concat([geneModules.loc[geneModules.GeneGroup.isin(["0","1","2","4","6","7"]),], geneModules.loc[geneModules.GeneGroup.isin(["3","5"]),]])

geneModulesNew = pd.concat([geneModules.loc[geneModules.GeneGroup=="3",], geneModules.loc[geneModules.GeneGroup=="6",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="7",]])

geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="2",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="4",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="0",]])

geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="1",]])
geneModulesNew = pd.concat([geneModulesNew, geneModules.loc[geneModules.GeneGroup=="5",]])


geneModulesNew


In [ ]:
geneModules = geneModulesNew

In [ ]:
geneCov = pd.DataFrame(np.corrcoef(coefsAll.transpose()), index=coefsAll.columns, columns=coefsAll.columns)
geneCov = geneCov.loc[geneModules.GeneName,geneModules.GeneName]

In [ ]:
geneModules.GeneGroup.value_counts()

In [ ]:
sns.clustermap(geneCov, row_cluster=False, col_cluster= False, cmap=plt.cm.RdBu, vmin=-0.5, vmax=0.5, row_colors=geneModules.GeneColor, col_colors=geneModules.GeneColor)


In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111)
cax = ax.matshow(geneCov, cmap=plt.cm.RdBu, vmin=-0.5, vmax=0.5)
#fig.colorbar(cax, orientation="horizontal")

In [ ]:
geneModules.to_csv("/home/beraslan/jovian-work/analysisSingle/ME_GeneModules_leiden_"+str(nClust)+"_Modules.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.matshow(coefsAll.loc[guideCov.index, geneCov.index], cmap=plt.cm.coolwarm, vmin=-0.2, vmax=0.2)


In [ ]:
sns.clustermap(coefsAll.loc[guideCov.index, geneCov.index], row_cluster=False, col_cluster= False, cmap=plt.cm.coolwarm, vmin=-0.2, vmax=0.2, figsize=(18,14))


In [ ]:
# koGenesAnnDat.obs['clusCol'] = koGenesAnnDat.obs['leiden'].astype("category").cat.codes

# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 0] = "#FF00FF" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 1] = "#008080" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 2] = "#0000FF" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 3] = "#FF0000" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 4] = "#008000" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 5] = "#00FF00" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 6] = "#00FFFF" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 7] = "#011000" 
# koGenesAnnDat.obs['clusCol'][koGenesAnnDat.obs['clusCol'] == 8] = "#011100" 

In [ ]:
# umap_combo = dict(zip(koGenesAnnDat.obs['KOGenes'], koGenesAnnDat.obsm['X_umap']))
# col_dict = dict(zip(koGenesAnnDat.obs['KOGenes'], koGenesAnnDat.obs['clusCol']))
# atomic_drugs = list(umap_combo.keys())

# f, ax = plt.subplots(figsize=(20, 20))
# ax.grid('off')
# ax.axis('off')
# for i, drug in enumerate(atomic_drugs):
#     ax.scatter(umap_combo[drug][0], umap_combo[drug][1], alpha=0.9, s=6, color=col_dict[drug])
    
    
# texts = []

# for l, drug in enumerate(atomic_drugs):
        
#     texts.append(ax.text(umap_combo[drug][0], umap_combo[drug][1], 
#                                     drug, fontsize=13, color=col_dict[drug]))
    
# adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.1))

# plt.show()   